# 02 - Modelo LSTM para series temporales electorales

Entrena una red LSTM para predecir el voto normalizado de un partido/coalición en cada entidad usando ventanas temporales.

**Nota metodológica:** se usa validación temporal: entrenar con elecciones antiguas, evaluar en las más recientes.

In [ ]:
import pandas as pd
import numpy as np
import os

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    print('TensorFlow', tf.__version__)
except ImportError:
    !pip install -q tensorflow
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout

from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

## Cargar y preparar panel

In [ ]:
BASE = '../data/processed/series'
# BASE = '/content/drive/MyDrive/inteligencia-electoral-mexico/data/processed/series'

df = pd.read_csv(os.path.join(BASE, 'serie_presidencia_entidad.csv'))
df['VOTOS'] = pd.to_numeric(df['VOTOS'], errors='coerce').fillna(0)
df['TOTAL_VOTOS'] = pd.to_numeric(df['TOTAL_VOTOS'], errors='coerce').fillna(1)
df['VOTO_NORMALIZADO'] = df['VOTOS'] / df['TOTAL_VOTOS']

# Seleccionar top partidos/coaliciones por varianza temporal
var = df.groupby('PARTIDO_COALICION')['VOTO_NORMALIZADO'].std().sort_values(ascending=False)
top_partidos = var.head(6).index.tolist()
print('Partidos seleccionados:', top_partidos)
df = df[df['PARTIDO_COALICION'].isin(top_partidos)]

## Construir matrices por entidad

In [ ]:
anios = sorted(df['ANIO'].unique())

panels = []
for entidad, g in df.groupby('ENTIDAD'):
    g = g.sort_values(['ANIO', 'PARTIDO_COALICION'])
    pivot = g.pivot(index='ANIO', columns='PARTIDO_COALICION', values='VOTO_NORMALIZADO')
    pivot = pivot.reindex(anios).fillna(0)
    panels.append((entidad, pivot.values))

X_all, y_all, entidades = [], [], []
WINDOW = 2
for entidad, mat in panels:
    if len(mat) < WINDOW + 1:
        continue
    for i in range(WINDOW, len(mat)):
        X_all.append(mat[i-WINDOW:i])
        y_all.append(mat[i])
        entidades.append(entidad)

X = np.stack(X_all)  # (samples, window, features)
y = np.stack(y_all)  # (samples, features)
print('Forma X:', X.shape, '| y:', y.shape)

## División temporal

In [ ]:
# Usar las últimas muestras (años más recientes) como test
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print('Train:', X_train.shape, 'Test:', X_test.shape)

## Entrenar LSTM

In [ ]:
tf.random.set_seed(42)
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(y_train.shape[1], activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=8,
                    validation_split=0.2, verbose=0,
                    callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)])
print('Mejor val_loss:', min(history.history['val_loss']))

## Evaluación

In [ ]:
y_pred = model.predict(X_test)
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))
print('MAE :', mean_absolute_error(y_test, y_pred))

plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title('Historial de entrenamiento LSTM')
plt.show()